# Fine-Tuning Qwen3 0.6B for IoT Applications

This notebook demonstrates how to fine-tune a small language model (Qwen3 0.6B) for IoT/Arduino/MQTT domain knowledge while preserving reasoning capabilities.

## Learning Objectives
- Understand LoRA (Low-Rank Adaptation) for efficient fine-tuning
- Work with instruction-tuning datasets
- Balance reasoning vs. direct-answer training examples
- Export models for deployment on resource-constrained devices (Raspberry Pi)

## Demo Mode vs Full Training

| Mode | Examples | Time (CPU) | Time (GPU) | Quality |
|------|----------|------------|------------|---------|
| **Quick Demo** | 10 | ~5-10 min | ~1 min | Low (proof of concept) |
| **Full Training** | 1000+ | 4-8 hours | ~15 min | Production-ready |

**This notebook defaults to Quick Demo mode** for presentations. For production results, use a cloud GPU (Google Colab, RunPod, etc.).

## Platform Notes

| Environment | Training Method | Notes |
|-------------|----------------|-------|
| **Docker/Linux** | Transformers+PEFT | Primary method for this demo |
| **macOS Native** | MLX | Best for Mac (won't work in Docker) |
| **Google Colab** | Unsloth | Free T4 GPU - use for full training |

**Important:** MLX requires native macOS - it won't work inside Docker containers.

---
## Section 1: Environment Setup

In [ ]:
# Detect environment
import platform
import subprocess
import sys

def detect_environment():
    """Detect the running environment and available hardware."""
    env = {
        'platform': platform.system(),
        'processor': platform.processor(),
        'in_docker': False,
        'has_cuda': False,
        'is_apple_silicon': False,
        'is_colab': False,
        'recommended_method': 'peft'  # Default
    }
    
    # Check if in Docker
    try:
        with open('/proc/1/cgroup', 'r') as f:
            env['in_docker'] = 'docker' in f.read()
    except:
        pass
    
    # Check for Colab
    try:
        import google.colab
        env['is_colab'] = True
        env['has_cuda'] = True
        env['recommended_method'] = 'unsloth'
    except ImportError:
        pass
    
    # Check for Apple Silicon (only works on native macOS, not Docker)
    if env['platform'] == 'Darwin' and 'arm' in env['processor'].lower() and not env['in_docker']:
        env['is_apple_silicon'] = True
        env['recommended_method'] = 'mlx'
    
    # Check for CUDA
    try:
        import torch
        env['has_cuda'] = torch.cuda.is_available()
        if env['has_cuda'] and not env['is_colab']:
            env['recommended_method'] = 'unsloth'
    except ImportError:
        pass
    
    return env

ENV = detect_environment()
print(f"Platform: {ENV['platform']}")
print(f"Processor: {ENV['processor']}")
print(f"In Docker: {ENV['in_docker']}")
print(f"Apple Silicon: {ENV['is_apple_silicon']}")
print(f"CUDA Available: {ENV['has_cuda']}")
print(f"Google Colab: {ENV['is_colab']}")
print(f"\n🎯 Recommended training method: {ENV['recommended_method'].upper()}")

In [ ]:
# Install dependencies based on environment
if ENV['recommended_method'] == 'mlx':
    print("Installing MLX dependencies (Apple Silicon)...")
    !pip install -q "mlx-lm[train]" datasets huggingface_hub
elif ENV['recommended_method'] == 'unsloth':
    print("Installing Unsloth dependencies (CUDA GPU)...")
    !pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
else:
    print("Installing Transformers+PEFT dependencies (CPU/Docker)...")
    !pip install -q transformers peft trl datasets accelerate torch huggingface_hub

---
## Section 2: Dataset Loading & Exploration

In [ ]:
from datasets import load_dataset
import json

# Option 1: Load from HuggingFace (140k Arduino/IoT conversations)
print("Loading Arduino/IoT dataset from HuggingFace...")
try:
    hf_dataset = load_dataset(
        "CJJones/Multiturn_Microcontroller-Arduino-LLM-Training",
        split="train"
    )
    print(f"Loaded {len(hf_dataset)} examples from HuggingFace")
    print(f"\nColumns: {hf_dataset.column_names}")
    print(f"\nFirst example:")
    print(hf_dataset[0])
except Exception as e:
    print(f"Could not load HuggingFace dataset: {e}")
    print("Using local sample dataset instead...")
    hf_dataset = None

In [ ]:
# Option 2: Load local sample dataset (50 curated IoT examples)
import os

# Find the sample dataset - check multiple possible locations
possible_paths = [
    "sample_iot_dataset.json",                      # Same directory
    "../finetune-qwen/sample_iot_dataset.json",     # If running from different folder
    "/home/jovyan/work/sample_iot_dataset.json",    # Docker container path
]

sample_path = None
for path in possible_paths:
    if os.path.exists(path):
        sample_path = path
        print(f"Found sample dataset at: {path}")
        break

if sample_path is None:
    raise FileNotFoundError(f"Could not find sample_iot_dataset.json. Searched: {possible_paths}")

with open(sample_path, 'r') as f:
    local_dataset = json.load(f)

print(f"Local dataset: {len(local_dataset['examples'])} examples")
print(f"\nDataset description: {local_dataset['description']}")
print(f"Split: {local_dataset['split']}")

# Show example types distribution
reasoning_count = sum(1 for ex in local_dataset['examples'] if ex['type'] == 'reasoning')
non_reasoning_count = len(local_dataset['examples']) - reasoning_count
print(f"\nReasoning examples: {reasoning_count}")
print(f"Non-reasoning examples: {non_reasoning_count}")

In [ ]:
# Explore example formats
print("=" * 60)
print("REASONING EXAMPLE (with <|thinking|> tags):")
print("=" * 60)
reasoning_ex = next(ex for ex in local_dataset['examples'] if ex['type'] == 'reasoning')
for msg in reasoning_ex['messages']:
    print(f"\n[{msg['role'].upper()}]")
    print(msg['content'][:500] + "..." if len(msg['content']) > 500 else msg['content'])

print("\n" + "=" * 60)
print("NON-REASONING EXAMPLE (direct answer):")
print("=" * 60)
non_reasoning_ex = next(ex for ex in local_dataset['examples'] if ex['type'] == 'non-reasoning')
for msg in non_reasoning_ex['messages']:
    print(f"\n[{msg['role'].upper()}]")
    print(msg['content'])

### Qwen3 Reasoning Format

Qwen3 supports **thinking mode** with special tokens:
- `/think` command enables reasoning (default)
- `/no_think` command disables reasoning
- `<|thinking|>...<|/thinking|>` tags wrap internal reasoning

**Training data should include ~75% reasoning examples** to preserve this capability at small model sizes.

---
## Section 3: Dataset Preparation

In [ ]:
def convert_to_qwen_format(examples):
    """
    Convert our dataset to Qwen3 chat format.
    
    Qwen3 uses:
    <|im_start|>system
    You are a helpful assistant.<|im_end|>
    <|im_start|>user
    Question<|im_end|>
    <|im_start|>assistant
    Answer<|im_end|>
    """
    formatted = []
    
    system_prompt = "You are an expert IoT and embedded systems assistant. You help developers with Arduino, ESP32, MQTT, LoRa, and sensor integration."
    
    for ex in examples:
        messages = ex['messages']
        
        # Build conversation string
        text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        
        for msg in messages:
            role = msg['role']
            content = msg['content']
            text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        
        formatted.append({
            'text': text,
            'type': ex.get('type', 'unknown')
        })
    
    return formatted

# Convert local dataset
formatted_data = convert_to_qwen_format(local_dataset['examples'])
print(f"Formatted {len(formatted_data)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(formatted_data[0]['text'][:500])

In [ ]:
# Create train/validation split
import random

random.seed(42)
shuffled = formatted_data.copy()
random.shuffle(shuffled)

split_idx = int(len(shuffled) * 0.9)
train_data = shuffled[:split_idx]
val_data = shuffled[split_idx:]

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

# Verify reasoning split in training data
train_reasoning = sum(1 for ex in train_data if ex['type'] == 'reasoning')
print(f"\nTraining set reasoning ratio: {train_reasoning}/{len(train_data)} ({train_reasoning/len(train_data)*100:.1f}%)")

In [ ]:
# Save formatted data for MLX (requires JSONL with 'messages' format)
import json
import os

os.makedirs('data', exist_ok=True)

# For MLX: save in messages format
def save_for_mlx(examples, filename):
    """Save in MLX-LM expected format."""
    with open(filename, 'w') as f:
        for ex in examples:
            # MLX expects {"messages": [{"role": "...", "content": "..."}]}
            # Find the original example
            orig_idx = formatted_data.index(ex)
            orig = local_dataset['examples'][orig_idx]
            
            mlx_entry = {
                "messages": [
                    {"role": "system", "content": "You are an expert IoT and embedded systems assistant."},
                ] + orig['messages']
            }
            f.write(json.dumps(mlx_entry) + '\n')

# For PEFT/Unsloth: save in text format
def save_for_peft(examples, filename):
    """Save in text format for SFTTrainer."""
    with open(filename, 'w') as f:
        for ex in examples:
            f.write(json.dumps({'text': ex['text']}) + '\n')

save_for_mlx(train_data, 'data/train.jsonl')
save_for_mlx(val_data, 'data/valid.jsonl')
save_for_peft(train_data, 'data/train_peft.jsonl')

print("Saved:")
print("  - data/train.jsonl (MLX format)")
print("  - data/valid.jsonl (MLX format)")
print("  - data/train_peft.jsonl (PEFT/Unsloth format)")

---
## Section 4: Model Loading

Choose the appropriate method based on your environment.

In [ ]:
# Method selection based on environment
METHOD = ENV['recommended_method']
print(f"Using method: {METHOD}")

if METHOD == 'mlx':
    print("\n⚠️ MLX training should be run from command line on native macOS.")
    print("See cell below for CLI commands.")
elif METHOD == 'peft':
    print("\n📦 Loading model with Transformers + PEFT (CPU-friendly)...")
elif METHOD == 'unsloth':
    print("\n🚀 Loading model with Unsloth (GPU-accelerated)...")

In [ ]:
# MLX Training (Apple Silicon Native ONLY - not in Docker)
if METHOD == 'mlx' and not ENV['in_docker']:
    print("""
    ╔══════════════════════════════════════════════════════════════╗
    ║  MLX Training Commands (run in native macOS terminal)         ║
    ╚══════════════════════════════════════════════════════════════╝
    
    # Quick training (100 iterations, ~10 min)
    mlx_lm.lora \
        --model Qwen/Qwen2.5-0.5B-Instruct \
        --train \
        --data ./data \
        --iters 100 \
        --batch-size 2 \
        --lora-layers 8
    
    # Test the fine-tuned model
    mlx_lm.generate \
        --model Qwen/Qwen2.5-0.5B-Instruct \
        --adapter-path adapters \
        --prompt "How do I connect an ESP32 to MQTT?"
    
    # Fuse adapter with base model
    mlx_lm.fuse \
        --model Qwen/Qwen2.5-0.5B-Instruct \
        --adapter-path adapters \
        --save-path fused_model
    
    # Export to GGUF for Raspberry Pi
    mlx_lm.convert --model fused_model --export-gguf -q Q4_K_M
    """)

In [ ]:
# PEFT Training (Works on CPU, Docker, AND Cloud GPUs!)
# This is the PRIMARY method - same code works everywhere

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Detect if GPU is available
USE_GPU = torch.cuda.is_available()
print(f"GPU Available: {USE_GPU}")

if USE_GPU:
    print("🚀 Running on GPU - full speed training!")
    device_map = "auto"
    dtype = torch.float16  # Use fp16 for GPU
else:
    print("🐢 Running on CPU - demo mode (slower)")
    device_map = "cpu"
    dtype = torch.float32

print(f"\nLoading {MODEL_NAME}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map=device_map,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Configure LoRA
# For quick demo: r=4, fewer modules
# For full training: r=16, all modules
DEMO_MODE = True  # Set to False for full training

if DEMO_MODE:
    print("\n⚡ DEMO MODE: Minimal LoRA for quick training")
    lora_config = LoraConfig(
        r=4,
        lora_alpha=8,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
else:
    print("\n🏋️ FULL MODE: Complete LoRA for best results")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\n✅ Model loaded with LoRA adapters")

In [ ]:
# Unsloth Training (Colab/CUDA GPU)
if METHOD == 'unsloth':
    from unsloth import FastModel
    
    model, tokenizer = FastModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
        max_seq_length=2048,
        load_in_4bit=True,
        dtype=None,
    )
    
    model = FastModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    
    print("✅ Model loaded with Unsloth optimizations")

---
## Section 5: Training

In [ ]:
# Training Configuration
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# === DEMO SETTINGS ===
# Adjust these for quick demo vs full training
DEMO_MODE = True  # Set False for full training
DEMO_EXAMPLES = 10  # Use only 10 examples for quick demo
MAX_SEQ_LENGTH = 256 if DEMO_MODE else 512

# Prepare dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='max_length'
    )

# Use subset for demo
if DEMO_MODE:
    train_texts = [ex['text'] for ex in train_data[:DEMO_EXAMPLES]]
    print(f"⚡ DEMO MODE: Using only {DEMO_EXAMPLES} examples")
else:
    train_texts = [ex['text'] for ex in train_data]
    print(f"🏋️ FULL MODE: Using all {len(train_data)} examples")

train_dataset = Dataset.from_dict({'text': train_texts})
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments - auto-detect GPU
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    per_device_train_batch_size=1 if not USE_GPU else 4,
    gradient_accumulation_steps=4 if not USE_GPU else 2,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    fp16=USE_GPU,  # Enable fp16 only on GPU
    use_cpu=not USE_GPU,
    no_cuda=not USE_GPU,
    report_to="none",
    warmup_steps=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print(f"\n📊 Training Configuration:")
print(f"   Examples: {len(train_dataset)}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Device: {'GPU' if USE_GPU else 'CPU'}")
print(f"   Estimated time: {'~1-2 min' if USE_GPU else '~5-10 min' if DEMO_MODE else '~4-8 hours'}")

In [ ]:
# Run Training!
import time

print("🏋️ Starting training...")
print("=" * 50)

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print("=" * 50)
print(f"✅ Training complete in {elapsed/60:.1f} minutes!")

# Save the adapter
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("💾 LoRA adapter saved to ./lora_adapter")

In [ ]:
# Training (Unsloth method)
if METHOD == 'unsloth':
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import Dataset
    
    # Prepare dataset
    train_texts = [ex['text'] for ex in train_data]
    train_dataset = Dataset.from_dict({'text': train_texts})
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=True,
            logging_steps=1,
            optim="adamw_8bit",
            output_dir="outputs",
        ),
    )
    
    print("🏋️ Starting Unsloth training...")
    trainer.train()
    print("✅ Training complete!")

---
## Section 6: Evaluation

In [ ]:
# Test prompts for evaluation
TEST_PROMPTS = [
    # Reasoning question (should trigger <|thinking|>)
    "How do I implement a temperature monitoring system with ESP32 and MQTT?",
    
    # Simple factual question
    "What is the default MQTT port?",
    
    # IoT-specific question
    "How can I reduce power consumption on a battery-powered Arduino sensor?",
]

In [ ]:
# Evaluate with PEFT model
if METHOD == 'peft':
    def generate_response(prompt, max_length=512):
        """Generate a response from the model."""
        full_prompt = f"<|im_start|>system\nYou are an expert IoT assistant.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer(full_prompt, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        # Extract just the assistant's response
        if "<|im_start|>assistant" in response:
            response = response.split("<|im_start|>assistant")[-1]
            if "<|im_end|>" in response:
                response = response.split("<|im_end|>")[0]
        return response.strip()
    
    print("Testing model responses (using BASE model, not fine-tuned):")
    print("=" * 60)
    for prompt in TEST_PROMPTS:
        print(f"\n📝 Prompt: {prompt}")
        print("-" * 40)
        response = generate_response(prompt, max_length=200)
        print(f"🤖 Response: {response[:300]}..." if len(response) > 300 else f"🤖 Response: {response}")

---
## Section 7: Export & Deployment to Raspberry Pi

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║            Deployment to Raspberry Pi                            ║
╚══════════════════════════════════════════════════════════════════╝

After training, export your model to GGUF format for Raspberry Pi:

══════════════════════════════════════════════════════════════════
OPTION 1: From MLX (Apple Silicon)
══════════════════════════════════════════════════════════════════

# Fuse LoRA adapter with base model
mlx_lm.fuse \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --adapter-path adapters \
    --save-path fused_model

# Convert to GGUF (Q4_K_M is good balance of quality/size)
mlx_lm.convert --model fused_model --export-gguf -q Q4_K_M

══════════════════════════════════════════════════════════════════
OPTION 2: From PEFT/Unsloth
══════════════════════════════════════════════════════════════════

# Merge LoRA adapter with base model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = PeftModel.from_pretrained(base_model, "./lora_adapter")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model")

# Then use llama.cpp to convert:
# python llama.cpp/convert_hf_to_gguf.py merged_model --outtype q4_k_m

══════════════════════════════════════════════════════════════════
RASPBERRY PI DEPLOYMENT
══════════════════════════════════════════════════════════════════

1. Copy the .gguf file to your Raspberry Pi

2. Install Ollama:
   curl -fsSL https://ollama.com/install.sh | sh

3. Create a Modelfile:
   FROM ./qwen-iot-Q4_K_M.gguf
   SYSTEM "You are an IoT and embedded systems expert."
   PARAMETER temperature 0.7

4. Import into Ollama:
   ollama create iot-assistant -f Modelfile

5. Run:
   ollama run iot-assistant "How do I connect to MQTT?"

Expected performance on Raspberry Pi 4 (4GB):
- Model size: ~400MB (Q4_K_M)
- Inference: ~5-10 tokens/second
- RAM usage: ~600MB
""")

---
## Summary

### What We Learned

1. **LoRA Fine-Tuning**: Efficient method that trains only ~1% of model parameters
2. **Reasoning Preservation**: Using 75% reasoning / 25% direct examples maintains thinking capability
3. **Platform Options**:
   - MLX for Apple Silicon (fastest local option)
   - Transformers+PEFT for Docker/CPU (slower but works anywhere)
   - Unsloth for GPU (fastest overall)
4. **Deployment**: GGUF quantization enables running on Raspberry Pi

### Next Steps

- [ ] Expand dataset to 500-1000 examples for better results
- [ ] Add more MQTT/LoRa specific examples
- [ ] Test on actual Raspberry Pi hardware
- [ ] Benchmark inference speed vs. base model